# TelcoRetain - Churn Prediction Model Training

This notebook trains a churn prediction model using the IBM Telco Customer Churn dataset.
It exports standardized artifacts for use by the TelcoRetain inference backend.

**Sections:**
1. Setup & Imports
2. Data Loading
3. Data Cleaning
4. Exploratory Data Analysis
5. Feature Engineering
6. Categorical Encoding
7. Train-Test Split
8. Model Training
9. Model Comparison
10. Hyperparameter Tuning
11. SHAP Analysis
12. Final Evaluation
13. Artifact Export

## 1. Setup & Imports

In [ ]:
# Install required packages (for Google Colab)
!pip install -q xgboost lightgbm shap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap
import joblib
import json
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

## 2. Data Loading

In [ ]:
# Load the IBM Telco Customer Churn dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumns ({len(df.columns)}):')
print(df.columns.tolist())
print(f'\nDataset Info:')
df.info()
print(f'\nFirst 5 rows:')
df.head()

## 3. Data Cleaning

In [ ]:
# Drop customerID - not useful for prediction
df = df.drop('customerID', axis=1)

# Convert TotalCharges to numeric (contains empty strings)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill TotalCharges nulls with median
total_charges_median = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(total_charges_median)
print(f'TotalCharges nulls filled with median: {total_charges_median:.2f}')

# Encode Churn target: No -> 0, Yes -> 1
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

# Validate no nulls remain
null_counts = df.isnull().sum()
assert null_counts.sum() == 0, f'Nulls remain: {null_counts[null_counts > 0]}'
print(f'\nNull validation passed - no missing values.')
print(f'Dataset shape after cleaning: {df.shape}')
print(f'Churn distribution:\n{df["Churn"].value_counts()}')

## 4. Exploratory Data Analysis

In [ ]:
# Churn distribution plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn distribution
sns.countplot(x='Churn', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Churn Distribution')
axes[0].set_xticklabels(['No (0)', 'Yes (1)'])

# MonthlyCharges distribution by churn
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True, ax=axes[1], palette='Set2')
axes[1].set_title('MonthlyCharges Distribution by Churn')

# Tenure distribution by churn
sns.histplot(data=df, x='tenure', hue='Churn', kde=True, ax=axes[2], palette='Set2')
axes[2].set_title('Tenure Distribution by Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap - Numeric Features')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# CLV: Customer Lifetime Value = MonthlyCharges * tenure
df['CLV'] = df['MonthlyCharges'] * df['tenure']

# HighValueCustomer: 1 if MonthlyCharges > 80, else 0
df['HighValueCustomer'] = (df['MonthlyCharges'] > 80).astype(int)

# TenureGroup: binned tenure into groups 1-4
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=[1, 2, 3, 4]
).astype(int)

print('Feature engineering complete.')
print(f'New features: CLV, HighValueCustomer, TenureGroup')
print(f'\nCLV stats:\n{df["CLV"].describe()}')
print(f'\nHighValueCustomer distribution:\n{df["HighValueCustomer"].value_counts()}')
print(f'\nTenureGroup distribution:\n{df["TenureGroup"].value_counts().sort_index()}')

## 6. Categorical Encoding

In [ ]:
# LabelEncode all object (categorical) columns
label_encoders = {}
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f'Categorical columns to encode ({len(categorical_cols)}):')
print(categorical_cols)

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f'  {col}: {list(le.classes_)}')

print(f'\nEncoding complete. {len(label_encoders)} encoders saved.')

## 7. Train-Test Split

In [ ]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# 80/20 split, stratified on target, random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTarget distribution (train):\n{y_train.value_counts(normalize=True)}')
print(f'\nTarget distribution (test):\n{y_test.value_counts(normalize=True)}')
print(f'\nFeature columns ({len(X.columns)}): {X.columns.tolist()}')

## 8. Model Training

In [ ]:
# Define models
models = {
    'LogisticRegression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        random_state=42, eval_metric='logloss', verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1
    )
}

# Train all models
trained_models = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f'  {name} training complete.')

print(f'\nAll {len(trained_models)} models trained successfully.')

## 9. Model Comparison

In [ ]:
# Evaluate all models
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

# Display comparison table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('Model Comparison (sorted by ROC-AUC):\n')
print(results_df.to_string(index=False))

# Select best model by ROC-AUC
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
best_roc_auc = results_df.iloc[0]['ROC-AUC']
print(f'\nBest model: {best_model_name} (ROC-AUC: {best_roc_auc:.4f})')

## 10. Hyperparameter Tuning

In [ ]:
# Define parameter grids for each model type
param_grids = {
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2']
    },
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5]
    },
    'XGBoost': {
        'n_estimators': [200, 300, 400],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1]
    },
    'LightGBM': {
        'n_estimators': [200, 300, 400],
        'max_depth': [-1, 10, 20],
        'learning_rate': [0.01, 0.05, 0.1]
    }
}

# Get appropriate param grid for best model
param_grid = param_grids[best_model_name]
print(f'Tuning {best_model_name} with GridSearchCV...')
print(f'Parameter grid: {param_grid}')

# Run GridSearchCV
grid_search = GridSearchCV(
    estimator=models[best_model_name],
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

# Get best model
tuned_model = grid_search.best_estimator_
print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.4f}')

# Evaluate tuned model on test set
y_pred_tuned = tuned_model.predict(X_test)
y_proba_tuned = tuned_model.predict_proba(X_test)[:, 1]
tuned_roc_auc = roc_auc_score(y_test, y_proba_tuned)
print(f'Tuned model test ROC-AUC: {tuned_roc_auc:.4f} (was {best_roc_auc:.4f})')

## 11. SHAP Analysis

In [ ]:
# SHAP analysis on the tuned model
print(f'Computing SHAP values for {best_model_name}...')

# Use TreeExplainer for tree-based models
if best_model_name in ['RandomForest', 'XGBoost', 'LightGBM']:
    explainer = shap.TreeExplainer(tuned_model)
else:
    explainer = shap.LinearExplainer(tuned_model, X_train)

shap_values = explainer.shap_values(X_test)

# Handle different SHAP output formats
if isinstance(shap_values, list):
    shap_values_plot = shap_values[1]  # Class 1 (churn)
else:
    shap_values_plot = shap_values

print('SHAP values computed successfully.')

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_plot, X_test, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Churn')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance bar chart (mean absolute SHAP values)
mean_shap = np.abs(shap_values_plot).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance.head(15), x='Mean |SHAP|', y='Feature', palette='viridis')
plt.title('Top 15 Features by Mean |SHAP Value|')
plt.tight_layout()
plt.show()

print('Top 10 most important features:')
print(feature_importance.head(10).to_string(index=False))

## 12. Final Evaluation

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name} (Tuned)')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print(f'Classification Report - {best_model_name} (Tuned):\n')
print(classification_report(y_test, y_pred_tuned, target_names=['No Churn', 'Churn']))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba_tuned)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'{best_model_name} (AUC = {tuned_roc_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Tuned Model')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Artifact Export

In [ ]:
import os

# Export path for artifacts
export_path = '../backend/ml/models/'
os.makedirs(export_path, exist_ok=True)

# 1. Save the tuned model
model_path = os.path.join(export_path, 'churn_model.pkl')
joblib.dump(tuned_model, model_path)
print(f'Saved: {model_path}')

# 2. Save label encoders
encoders_path = os.path.join(export_path, 'label_encoders.pkl')
joblib.dump(label_encoders, encoders_path)
print(f'Saved: {encoders_path}')

# 3. Save feature columns
feature_columns_path = os.path.join(export_path, 'feature_columns.pkl')
feature_columns = X.columns.tolist()
joblib.dump(feature_columns, feature_columns_path)
print(f'Saved: {feature_columns_path}')

# 4. Save model metadata
metadata = {
    'model_type': best_model_name,
    'training_date': datetime.now().isoformat(),
    'dataset_name': 'IBM Telco Customer Churn',
    'feature_count': len(feature_columns),
    'performance_metrics': {
        'accuracy': float(accuracy_score(y_test, y_pred_tuned)),
        'precision': float(precision_score(y_test, y_pred_tuned)),
        'recall': float(recall_score(y_test, y_pred_tuned)),
        'f1_score': float(f1_score(y_test, y_pred_tuned)),
        'roc_auc': float(tuned_roc_auc)
    },
    'class_distribution': {
        'no_churn': int((y == 0).sum()),
        'churn': int((y == 1).sum())
    }
}

metadata_path = os.path.join(export_path, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Saved: {metadata_path}')

print(f'\n--- Export Summary ---')
print(f'Model type: {best_model_name}')
print(f'Feature count: {len(feature_columns)}')
print(f'ROC-AUC: {tuned_roc_auc:.4f}')
print(f'Export directory: {export_path}')
print(f'\nAll artifacts exported successfully.')